In [ ]:
import mne
import scipy
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import pickle
import utils
import algo
import os
import random
import glob
import copy
import itertools
from tqdm import tqdm
from numpy import linalg as LA
from scipy.stats import zscore, pearsonr
from scipy.io import savemat, loadmat
from scipy import signal
%matplotlib widget

In [ ]:
def prepare_data_subj(Subj_ID, fs):
    eeg_list, eog_list, gaze_list, feats_list = utils.load_subj(Subj_ID)
    gaze_coords_list = [gaze[:,0:2,:] for gaze in gaze_list]
    saccade_list = [np.expand_dims(gaze[:,2,:], axis=1) for gaze in gaze_list]
    blink_list = [np.expand_dims(gaze[:,3,:], axis=1) for gaze in gaze_list]
    saccade_list = utils.refine_saccades(saccade_list, blink_list)
    gaze_velocity_list = [utils.calcu_gaze_velocity(gaze) for gaze in gaze_list]
    objflow_list = [np.expand_dims(feats[:,8,:], axis=1) for feats in feats_list]
    eeg_reg_list = [utils.regress_out(eeg, eog) for eeg, eog in zip(eeg_list, eog_list)]
    eeg_list = [utils.remove_shot_cuts_and_center(d, fs, remove_time=1) for d in eeg_list]
    eog_list = [utils.remove_shot_cuts_and_center(d, fs, remove_time=1) for d in eog_list]
    gaze_coords_list = [utils.remove_shot_cuts_and_center(d, fs, remove_time=1) for d in gaze_coords_list]
    saccade_list = [utils.remove_shot_cuts_and_center(d, fs, remove_time=1, CENTER=False) for d in saccade_list]
    blink_list = [utils.remove_shot_cuts_and_center(d, fs, remove_time=1, CENTER=False) for d in blink_list]
    gaze_velocity_list = [utils.remove_shot_cuts_and_center(d, fs, remove_time=1) for d in gaze_velocity_list]
    objflow_list = [utils.remove_shot_cuts_and_center(d, fs, remove_time=1) for d in objflow_list]
    eeg_reg_list = [utils.remove_shot_cuts_and_center(d, fs, remove_time=1) for d in eeg_reg_list]
    data_multitask_dict = {'EEG': eeg_list, 'EOG': eog_list, 'GAZE': gaze_coords_list, 'GAZE_V': gaze_velocity_list, 'EEG-EOG': eeg_reg_list}
    return data_multitask_dict, objflow_list, saccade_list, blink_list

def find_most_correlated_segment(series_a, series_b):
    len_a = len(series_a)
    len_b = len(series_b)
    if len_b > len_a:
        raise ValueError("Series B cannot be longer than series A")
    # Calculate correlations for all possible segments
    correlations = []
    for i in range(len_a - len_b + 1):
        segment = series_a[i:i + len_b]
        corr, p_value = pearsonr(segment, series_b)

        correlations.append({
            'correlation': corr,
            'p_value': p_value,
            'start_idx': i,
            'end_idx': i + len_b,
            'segment': segment.copy()
        })
    # Find the segment with highest absolute correlation
    best_match = max(correlations, key=lambda x: abs(x['correlation']))
    return {
        'best_correlation': best_match['correlation'],
        'p_value': best_match['p_value'],
        'start_index': best_match['start_idx'],
        'end_index': best_match['end_idx'],
        'best_segment': best_match['segment'],
        'all_correlations': [c['correlation'] for c in correlations]
    }

def plot_correlation_results(series_a, series_b, result):
    """
    Plot the original series and highlight the best matching segment.
    """
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 10))
    
    # Plot full series A with highlighted segment
    ax1.plot(series_a, label='Series A', alpha=0.7)
    start_idx = result['start_index']
    end_idx = result['end_index']
    ax1.plot(range(start_idx, end_idx), result['best_segment'], 
             color='red', linewidth=2, label=f'Best segment (r={result["best_correlation"]:.3f})')
    ax1.set_title('Series A with Best Matching Segment Highlighted')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot correlation along the sliding window
    ax2.plot(result['all_correlations'])
    ax2.axhline(y=result['best_correlation'], color='red', linestyle='--', 
                label=f'Best correlation: {result["best_correlation"]:.3f}')
    ax2.axvline(x=start_idx, color='red', linestyle='--', alpha=0.5)
    ax2.set_title('Correlation Along Sliding Window')
    ax2.set_xlabel('Starting Position')
    ax2.set_ylabel('Correlation')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Compare best segment with series B
    ax3.plot(result['best_segment'], label='Best segment from A', marker='o', markersize=3)
    ax3.plot(series_b, label='Series B', marker='s', markersize=3)
    ax3.set_title(f'Comparison: Best Segment vs Series B (r={result["best_correlation"]:.3f})')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
subjects = ['AS', 'YY', 'CM', 'SUB1', 'IR', 'SUB2', 'YZ', 'WD', 'CC', 'CW', 'WS', 'VC','HV','JC','DV','CD','JV','KY','KB','SC']
# load the data of the first protocol
fs = 30
with open('data/1stprotocol/features.pkl', 'rb') as f:
    features_list = pickle.load(f)
with open('data/1stprotocol/eeg_multisub.pkl', 'rb') as f:
    eeg_multisub_list = pickle.load(f)
with open('data/1stprotocol/eog_multisub.pkl', 'rb') as f:
    eog_multisub_list = pickle.load(f)
with open('data/1stprotocol/hf_multisub.pkl', 'rb') as f:
    hf_multisub_list = pickle.load(f)

In [ ]:
video_idx = [2, 3, 5, 6, 8, 10, 12]
eeg_list = [eeg_multisub_list[vid] for vid in video_idx]
eog_list = [eog_multisub_list[vid] for vid in video_idx]
objflow_list = [features_list[vid][:,8] for vid in video_idx]

In [ ]:
_, objflow_ref_list, _, _ = prepare_data_subj(0, fs)

In [ ]:
for i in range(len(video_idx)):
    result = find_most_correlated_segment(objflow_list[i], objflow_ref_list[i][:,0,0])
    eeg_list[i] = eeg_list[i][result['start_index']:result['end_index'], ...]
    eog_list[i] = eog_list[i][result['start_index']:result['end_index'], ...]
    objflow_list[i] = objflow_ref_list[i][:,0,0]


In [ ]:
eeg_reg_list = [utils.regress_out(eeg, eog) for eeg, eog in zip(eeg_list, eog_list)]
eeg_list = [utils.remove_shot_cuts_and_center(d, fs, remove_time=0) for d in eeg_list]
eog_list = [utils.remove_shot_cuts_and_center(d, fs, remove_time=0) for d in eog_list]
objflow_list = [utils.remove_shot_cuts_and_center(d, fs, remove_time=0) for d in objflow_list]
eeg_reg_list = [utils.remove_shot_cuts_and_center(d, fs, remove_time=0) for d in eeg_reg_list]

In [ ]:
eeg_reg_list = [eeg_reg[:,:,:, np.newaxis] for eeg_reg in eeg_reg_list]
objflow_list = [objflow[:, np.newaxis, np.newaxis] for objflow in objflow_list]

In [ ]:
L_EEG = 3
L_Stim = 15
offset_EEG = 1
offset_Stim = 0

RUN_TIMES = 5
MOD = 'EEG-EOG'
trial_len = 45
task_train = [1]
nb_nearby_samples = None # [9, 3]
BOOTSTRAP = True
n_components = 5 if (MOD != 'GAZE_V' and MOD != 'GAZE') else 3
range_into_account = 3
nb_comp_into_account = 2

In [ ]:
def analyze_all(eeg_reg_list, objflow_list, fs, L_EEG, L_Stim, offset_EEG, offset_Stim, range_into_account, nb_comp_into_account, task_train=[1], trial_len=30, n_components=5, save_name=None, MOD='EEG-EOG', PERMU_TEST=False, BOOTSTRAP=True, nb_nearby_samples=None):
    all_acc = []
    corr_match_all_subj = []
    corr_mismatch_all_subj = []
    acc_permu_all = []
    start_points = None
    for Subj_ID in range(len(subjects)):
        print(f"###################\nSubject {Subj_ID + 1} / {len(subjects)}")
        data_list = [eeg_reg[:,:,Subj_ID,:] for eeg_reg in eeg_reg_list]
        data_masked_list = None
        objflow_masked_list = None
        CCA = algo.CanonicalCorrelationAnalysis(data_list, objflow_list, fs, L_EEG, L_Stim, offset_EEG, offset_Stim, task_train=task_train, leave_out=1, n_components=n_components, EEG_masked=data_masked_list, Stim_masked=objflow_masked_list)
        corr_match_data, corr_mismatch_data, acc_permu_list, start_points, _ = CCA.match_mismatch(trial_len=trial_len, PERMU_TEST=PERMU_TEST, BOOTSTRAP=BOOTSTRAP, given_start_points=start_points)
        print("###########Match-Mismatch, TASK 1, 2, 3###########")
        acc_all_tasks, _, _, _, _ = utils.eval_compete_3D(corr_match_data, corr_mismatch_data, True, range_into_account=range_into_account, nb_comp_into_account=nb_comp_into_account, message=True)
        all_acc.append({
            'Subject': Subj_ID + 1,
            'Task_1': acc_all_tasks[0],
        })
        corr_match_all_subj.append(corr_match_data)
        corr_mismatch_all_subj.append(corr_mismatch_data)
        if PERMU_TEST:
            acc_permu_all += acc_permu_list
    all_acc = pd.DataFrame(all_acc)
    if PERMU_TEST:
        acc_permu = np.concatenate(acc_permu_all, axis=0)
        alpha = 0.05
        lower_bound = np.percentile(acc_permu, alpha/2*100)
        upper_bound = np.percentile(acc_permu, (1-alpha/2)*100)
    else:
        lower_bound = None
        upper_bound = None
    # add two columns to all_acc for lower and upper bound
    all_acc['lower_bound'] = lower_bound
    all_acc['upper_bound'] = upper_bound
    if save_name is not None:
        save_path = f"tables/{MOD}/{save_name}"
        all_acc.to_csv(f"{save_path}_acc_{trial_len}_train_{task_train}{'_BT' if BOOTSTRAP else ''}{('_nearby'+str(nb_nearby_samples)) if nb_nearby_samples is not None else ''}_1stprotocol.csv", index=False)
        # save corr_match_all_subj, corr_mismatch_all_subj, start_idx as dictionary
        corr_res = {
            'corr_match_all_subj': corr_match_all_subj,
            'corr_mismatch_all_subj': corr_mismatch_all_subj,
            'start_points': start_points
        }
        # save as pickle file
        with open(f"{save_path}_corr_{trial_len}_train_{task_train}{'_BT' if BOOTSTRAP else ''}{('_nearby'+str(nb_nearby_samples)) if nb_nearby_samples is not None else ''}_1stprotocol.pickle", 'wb') as f:
            pickle.dump(corr_res, f)
    return all_acc, corr_match_all_subj, corr_mismatch_all_subj, start_points

In [ ]:
# create a dictionary to store the results
for i in range(RUN_TIMES):
    save_name = f"RUN_{i+1}"
    PERMU_TEST = (i == 0)
    all_acc, corr_match_all_subj, corr_mismatch_all_subj, start_idx = analyze_all(eeg_reg_list, objflow_list, fs, L_EEG, L_Stim, offset_EEG, offset_Stim, range_into_account, nb_comp_into_account, task_train=[1], trial_len=trial_len, n_components=n_components, save_name=save_name, MOD='EEG-EOG', PERMU_TEST=PERMU_TEST, BOOTSTRAP=True, nb_nearby_samples=nb_nearby_samples)
    print(all_acc)